In [ ]:
# install the required libraries
! pip install sacremoses # tokenization
! pip install sentence-splitter
! pip install transformers
! pip install SentencePiece

In [ ]:
import torch
from transformers import PegasusForConditionalGeneration, PegasusTokenizer

In [ ]:
model = PegasusForConditionalGeneration.from_pretrained("tuner007/pegasus_paraphrase")
tokenizer = PegasusTokenizer.from_pretrained('tuner007/pegasus_paraphrase')

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at tuner007/pegasus_paraphrase and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
text = "The ultimate test of your knowledge is your capacity to convey it to another."
batch = tokenizer([text], padding=True, truncation=True, max_length=60, return_tensors='pt')
output = model.generate (**batch, max_length=60, num_beams=5, num_return_sequences=5, temperature=1.5)

In [ ]:
results = tokenizer.batch_decode(output, skip_special_tokens=True)
results

['The test of your knowledge is your ability to convey it.',
 'The ability to convey your knowledge is the ultimate test of your knowledge.',
 'The test of your knowledge is your ability to communicate it.',
 'Your capacity to convey your knowledge is the ultimate test of your knowledge.',
 'The test of your knowledge is how well you can convey it.']

In [ ]:
# Save model and tokinizer
model.save_pretrained("/content/drive/MyDrive/model")
tokenizer.save_pretrained("/content/drive/MyDrive/tokenizer")

/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3339: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 60, 'num_beams': 8, 'length_penalty': 0.8}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


('/content/drive/MyDrive/tokenizer/tokenizer_config.json',
 '/content/drive/MyDrive/tokenizer/special_tokens_map.json',
 '/content/drive/MyDrive/tokenizer/spiece.model',
 '/content/drive/MyDrive/tokenizer/added_tokens.json')

In [ ]:
def get_response(input_text, num_return_sequences=5, num_beams=5):
  batch = tokenizer([text], padding=True, truncation=True, max_length=60, return_tensors='pt')
  translated = model.generate (**batch, max_length=60, num_beams=num_beams, num_return_sequences=num_return_sequences, temperature=1.5)
  tgt_text = tokenizer.batch_decode(translated, skip_special_tokens=True)
  return tgt_text

In [ ]:
num_beams=10
num_return_sequences=10
context = "Machine learning is growing field in AI"
get_response(context, num_return_sequences, num_beams)

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1.5` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


['The test of your knowledge is your ability to convey it.',
 'The ability to convey your knowledge is the ultimate test of your knowledge.',
 'The ability to convey your knowledge is the most important test of your knowledge.',
 'Your capacity to convey your knowledge is the ultimate test of it.',
 'The test of your knowledge is your ability to communicate it.',
 'Your capacity to convey your knowledge is the ultimate test of your knowledge.',
 'Your capacity to convey your knowledge is the most important test of your knowledge.',
 'The test of your knowledge is how well you can convey it.',
 'Your capacity to convey your knowledge is the ultimate test.',
 'The ability to convey your knowledge is the ultimate test of it.']

In [ ]:
!pip install streamlit

In [ ]:
%%writefile app.py
import streamlit as st
from transformers import PegasusForConditionalGeneration, PegasusTokenizer
import torch

# Load model and tokenizer once
@st.cache_resource
def load_model():
    model = PegasusForConditionalGeneration.from_pretrained("tuner007/pegasus_paraphrase")
    tokenizer = PegasusTokenizer.from_pretrained("tuner007/pegasus_paraphrase")
    return model, tokenizer

model, tokenizer = load_model()

def get_response(input_text, num_return_sequences=5, num_beams=5):
    batch = tokenizer([input_text], padding=True, truncation=True, max_length=60, return_tensors='pt')
    translated = model.generate(**batch, max_length=60, num_beams=num_beams, num_return_sequences=num_return_sequences, temperature=1.5)
    tgt_text = tokenizer.batch_decode(translated, skip_special_tokens=True)
    return tgt_text

st.title("Paraphrase Generator with Image Interface")

# Display images as options for number of beams (example)
st.write("Select number of beams:")

col1, col2, col3 = st.columns(3)

with col1:
    if st.button("🔵 5 Beams"):
        st.session_state.num_beams = 5
with col2:
    if st.button("🟢 10 Beams"):
        st.session_state.num_beams = 10
with col3:
    if st.button("🔴 15 Beams"):
        st.session_state.num_beams = 15

num_beams = st.session_state.get("num_beams", 5)

# Display images as options for number of return sequences
st.write("Select number of return sequences:")

col1, col2, col3 = st.columns(3)

with col1:
    if st.button("🟡 3 Sequences"):
        st.session_state.num_return_sequences = 3
with col2:
    if st.button("🟠 5 Sequences"):
        st.session_state.num_return_sequences = 5
with col3:
    if st.button("🟣 10 Sequences"):
        st.session_state.num_return_sequences = 10

num_return_sequences = st.session_state.get("num_return_sequences", 5)

# Input text box
input_text = st.text_area("Enter text to paraphrase:", height=100)

if st.button("Generate Paraphrases"):
    if input_text.strip():
        with st.spinner("Generating..."):
            results = get_response(input_text, num_return_sequences=num_return_sequences, num_beams=num_beams)
        st.success("Done!")
        for i, res in enumerate(results, 1):
            st.write(f"Paraphrase {i}: {res}")
    else:
        st.error("Please enter some text to paraphrase.")

Overwriting app.py


In [ ]:
!pip install pyngrok

In [ ]:
!ngrok config add-authtoken 2x2MXAL8EsbSqwH4ilLliJEg4ML_7LJyxENQLcxMDLVyhL8Xz

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
! streamlit run app.py --server.address=0.0.0.0 --server.port=8501




  You can now view your Streamlit app in your browser.

  URL: http://0.0.0.0:8501

  Stopping...
  Stopping...


In [ ]:
from pyngrok import ngrok
import subprocess
import time
import socket
import signal
import sys

NGROK_AUTH_TOKEN = "from pyngrok import ngrok"
import subprocess
import time
import socket
import signal
import sys

NGROK_AUTH_TOKEN = "2x2MXAL8EsbSqwH4ilLliJEg4ML_7LJyxENQLcxMDLVyhL8Xz"

# Set ngrok auth token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Kill any existing tunnels/processes
ngrok.kill()

# Function to check if port is open
def is_port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        result = sock.connect_ex(('127.0.0.1', port))
        return result == 0

# Start Streamlit app with proper options
streamlit_process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Wait until port 8501 is open or timeout after 60 seconds
timeout = 60
start_time = time.time()
while not is_port_open(8501):
    if time.time() - start_time > timeout:
        print("Timeout waiting for Streamlit to start.")
        streamlit_process.terminate()
        sys.exit(1)
    time.sleep(1)

# Open ngrok tunnel
public_url = ngrok.connect(8501)
print(f"Streamlit app URL: {public_url}")

def signal_handler(sig, frame):
    print("Stopping Streamlit and ngrok...")
    streamlit_process.terminate()
    ngrok.kill()
    sys.exit(0)

signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)

# Stream Streamlit logs to console
try:
    while True:
        output = streamlit_process.stdout.readline()
        if output == '' and streamlit_process.poll() is not None:
            break
        if output:
            print(output.strip())
except KeyboardInterrupt:
    signal_handler(None, None)

# Set ngrok auth token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Kill any existing tunnels/processes
ngrok.kill()

# Function to check if port is open
def is_port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        result = sock.connect_ex(('127.0.0.1', port))
        return result == 0

# Start Streamlit app with proper options
streamlit_process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Wait until port 8501 is open or timeout after 60 seconds
timeout = 60
start_time = time.time()
while not is_port_open(8501):
    if time.time() - start_time > timeout:
        print("Timeout waiting for Streamlit to start.")
        streamlit_process.terminate()
        sys.exit(1)
    time.sleep(1)

# Open ngrok tunnel
public_url = ngrok.connect(8501)
print(f"Streamlit app URL: {public_url}")

def signal_handler(sig, frame):
    print("Stopping Streamlit and ngrok...")
    streamlit_process.terminate()
    ngrok.kill()
    sys.exit(0)

signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)

# Stream Streamlit logs to console
try:
    while True:
        output = streamlit_process.stdout.readline()
        if output == '' and streamlit_process.poll() is not None:
            break
        if output:
            print(output.strip())
except KeyboardInterrupt:
    signal_handler(None, None)

Streamlit app URL: NgrokTunnel: "https://78c5-34-139-207-95.ngrok-free.app" -> "http://localhost:8501"




ERROR:pyngrok.process.ngrok:t=2025-05-13T10:33:03+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Your account is limited to 1 simultaneous ngrok agent sessions.\nYou can run multiple simultaneous tunnels from a single agent session by defining the tunnels in your agent configuration file and starting them with the command `ngrok start --all`.\nRead more about the agent configuration file: https://ngrok.com/docs/secure-tunnels/ngrok-agent/reference/config\nYou can view your current agent sessions in the dashboard:\nhttps://dashboard.ngrok.com/agents\r\n\r\nERR_NGROK_108\r\n"
ERROR:pyngrok.process.ngrok:t=2025-05-13T10:33:03+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Your account is limited to 1 simultaneous ngrok agent sessions.\nYou can run multiple simultaneous tunnels from a single agent session by defining the tunnels in your agent configuration file and starting them with the command `ngrok st

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Your account is limited to 1 simultaneous ngrok agent sessions.\nYou can run multiple simultaneous tunnels from a single agent session by defining the tunnels in your agent configuration file and starting them with the command `ngrok start --all`.\nRead more about the agent configuration file: https://ngrok.com/docs/secure-tunnels/ngrok-agent/reference/config\nYou can view your current agent sessions in the dashboard:\nhttps://dashboard.ngrok.com/agents\r\n\r\nERR_NGROK_108\r\n.

In [ ]:
!streamlit run /usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py [ARGUMENTS]




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.139.207.95:8501

  Stopping...
  Stopping...
